# TalentPulse — Full ML Training Pipeline (`train_pipeline.py`)

Notebook conversion of `../train_pipeline.py` (original `.py` script is kept intact).

### Pipeline Stages
- **Stage 1:** From-Scratch Exploratory Data Analysis (EDA) & Legacy Outlier Corruption Audit
- **Stage 2:** Leak-Free 80/20 Stratified Split, Hardcoded Class Weights (`w0`, `w1`) for Logistic Regression & Random Forest, and Leak-Free CV Fold Caches
- **Stage 3:** Live Hyperparameter Optimization (HPO) across 6 diverse model families (`Logistic Regression`, `Random Forest`, `XGBoost`, `Extra Trees`, `Gradient Boosting`, `Calibrated RBF-SVC`)
- **Stage 4:** Ensemble Learning (`Core-3 Voting`, `Weighted Soft-Vote`, `Stacking`) & 10-Fold OOF Diversity Audit
- **Stage 5:** Out-of-Fold Decision Threshold Optimization (Balanced F1 & HR High-Recall $F_{1.5}$)
- **Stage 6:** Final Fit, Untouched Holdout Test Evaluation, Baseline Lift & Subgroup Fairness Audit
- **Stage 7:** Production Artifact Export (`../models/*.pkl`), All **11 Inline Feature & Model Comparison Plots**, and Automated Smoke Verification

## Setup: Imports, Paths & Pipeline Configuration

In [ ]:
from __future__ import annotations

import ctypes
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, Tuple

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import xgboost as xgb_lib
from joblib import Parallel, delayed
from scipy import stats
from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    fbeta_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterSampler, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Ensure project root is on sys.path so `feature_engineering.py` imports cleanly
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from feature_engineering import (
    LEGACY_18_FEATURES,
    WeightedSoftVotingClassifier,
    build_preprocessor,
)

# Notebook Configuration (equivalent to CLI args in train_pipeline.py)
args = SimpleNamespace(
    data=str(PROJECT_ROOT / 'data' / 'employee_attrition.csv'),
    models_dir=str(PROJECT_ROOT / 'models'),
    reports_dir=str(PROJECT_ROOT / 'reports'),
    hpo_folds=5,
    oof_folds=10,
    n_iter=30,
    seed=42,
    skip_plots=False,
    show_plots=True,  # Renders all 11 plots inline inside Jupyter!
)
print(f'Project Root: {PROJECT_ROOT}')
print(f'Config      : {args}')

## Helper 1: Live Terminal/Output Telemetry Renderer (`TerminalUI`)

In [ ]:
# =============================================================================
# ZERO-DEPENDENCY LIVE TERMINAL TELEMETRY UI
# =============================================================================
class TerminalUI:
    """Clean, ANSI-colored live terminal progress & telemetry renderer."""

    def __init__(self, use_color: Optional[bool] = None) -> None:
        self.is_tty = sys.stdout.isatty()
        if use_color is None:
            use_color = self._enable_windows_ansi()
        self.use_color = use_color
        self.start_time = time.perf_counter()

        # ANSI palette
        self.RESET = "\033[0m" if use_color else ""
        self.BOLD = "\033[1m" if use_color else ""
        self.DIM = "\033[2m" if use_color else ""
        self.CYAN = "\033[96m" if use_color else ""
        self.BLUE = "\033[94m" if use_color else ""
        self.GREEN = "\033[92m" if use_color else ""
        self.YELLOW = "\033[93m" if use_color else ""
        self.RED = "\033[91m" if use_color else ""
        self.MAGENTA = "\033[95m" if use_color else ""
        self.WHITE = "\033[97m" if use_color else ""

        # Box / bar glyphs (safe UTF-8 with ASCII fallback)
        enc = (getattr(sys.stdout, "encoding", "") or "").lower()
        unicode_ok = "utf" in enc
        self.BAR_FULL = "█" if unicode_ok else "#"
        self.BAR_EMPTY = "░" if unicode_ok else "-"
        self.LINE_H = "═" if unicode_ok else "="
        self.LINE_S = "─" if unicode_ok else "-"
        self.STAR = "★" if unicode_ok else "*"
        self.CHECK = "✓" if unicode_ok else "[OK]"
        self.WARN = "▲" if unicode_ok else "!"

    @staticmethod
    def _enable_windows_ansi() -> bool:
        if os.environ.get("NO_COLOR"):
            return False
        if os.name != "nt":
            return True
        try:
            kernel32 = ctypes.windll.kernel32
            handle = kernel32.GetStdHandle(-11)  # STD_OUTPUT_HANDLE
            mode = ctypes.c_uint32()
            if kernel32.GetConsoleMode(handle, ctypes.byref(mode)):
                kernel32.SetConsoleMode(handle, mode.value | 0x0004)
            return True
        except Exception:
            return False

    def elapsed(self) -> str:
        secs = time.perf_counter() - self.start_time
        return f"{secs:6.1f}s"

    def banner(self, title: str, subtitle: str) -> None:
        w = 94
        print(f"\n{self.CYAN}{self.BOLD}{self.LINE_H * w}{self.RESET}")
        print(f"{self.CYAN}{self.BOLD}  {title}{self.RESET}")
        print(f"{self.DIM}  {subtitle}{self.RESET}")
        print(f"{self.CYAN}{self.BOLD}{self.LINE_H * w}{self.RESET}", flush=True)

    def stage(self, step_num: int, total_steps: int, title: str) -> None:
        w = 94
        tag = f"[STAGE {step_num}/{total_steps}]"
        print(f"\n{self.BLUE}{self.BOLD}{self.LINE_S * w}{self.RESET}")
        print(
            f"{self.MAGENTA}{self.BOLD}{tag}{self.RESET} "
            f"{self.WHITE}{self.BOLD}{title}{self.RESET} "
            f"{self.DIM}(t+{self.elapsed()}){self.RESET}"
        )
        print(f"{self.BLUE}{self.BOLD}{self.LINE_S * w}{self.RESET}", flush=True)

    def subheader(self, title: str) -> None:
        print(f"\n{self.CYAN}{self.BOLD}>> {title}{self.RESET}", flush=True)

    def info(self, msg: str) -> None:
        print(f"   {msg}", flush=True)

    def success(self, msg: str) -> None:
        print(f"   {self.GREEN}{self.BOLD}{self.CHECK}{self.RESET} {msg}", flush=True)

    def warning(self, msg: str) -> None:
        print(f"   {self.YELLOW}{self.BOLD}{self.WARN}{self.RESET} {msg}", flush=True)

    def progress_bar(self, current: int, total: int, width: int = 24) -> str:
        frac = min(max(current / max(total, 1), 0.0), 1.0)
        filled = int(round(frac * width))
        return f"[{self.BAR_FULL * filled}{self.BAR_EMPTY * (width - filled)}]"

    def table(
        self,
        headers: List[str],
        rows: List[List[str]],
        alignments: Optional[List[str]] = None,
    ) -> None:
        if not rows:
            return
        widths = [len(h) for h in headers]
        for r in rows:
            for idx, cell in enumerate(r):
                widths[idx] = max(widths[idx], len(str(cell)))
        if alignments is None:
            alignments = ["<"] + [">"] * (len(headers) - 1)

        def fmt_row(vals: List[str]) -> str:
            return "   " + " | ".join(f"{str(v):{al}{w}}" for v, w, al in zip(vals, widths, alignments))

        sep = "   " + "-+-".join("-" * w for w in widths)
        print(f"{self.BOLD}{fmt_row(headers)}{self.RESET}")
        print(f"{self.DIM}{sep}{self.RESET}")
        for r in rows:
            print(fmt_row(r))
        sys.stdout.flush()

## Helper 2: From-Scratch EDA Engine (`run_eda_stage`)

In [ ]:
# =============================================================================
# STAGE 1: FROM-SCRATCH EXPLORATORY DATA ANALYSIS (EDA)
# =============================================================================
def run_eda_stage(
    df: pd.DataFrame,
    raw_path: Path,
    legacy_clean_path: Path,
    ui: TerminalUI,
) -> Dict[str, Any]:
    ui.stage(1, 7, "From-Scratch Exploratory Data Analysis (EDA) & Data Quality Audit")

    n_rows, n_cols = df.shape
    n_dups = int(df.duplicated().sum())
    n_missing = int(df.isnull().sum().sum())
    y = (df["Attrition"] == "Yes").astype(int)
    n_pos = int(y.sum())
    n_neg = int((y == 0).sum())
    pos_rate = float(y.mean() * 100.0)
    neg_rate = float((1.0 - y.mean()) * 100.0)
    imbalance_ratio = n_neg / max(n_pos, 1)

    ui.info(f"Source Dataset   : {raw_path} ({n_rows:,} rows x {n_cols} columns)")
    ui.info(f"Data Integrity   : Missing values = {n_missing} | Duplicate rows = {n_dups}")
    bar_no = ui.progress_bar(n_neg, n_rows, width=22)
    bar_yes = ui.progress_bar(n_pos, n_rows, width=22)
    ui.info(f"Target [No  = 0] : {bar_no} {n_neg:4d} ({neg_rate:5.2f}%)")
    ui.info(f"Target [Yes = 1] : {bar_yes} {n_pos:4d} ({pos_rate:5.2f}%)  [Imbalance Ratio: {imbalance_ratio:.2f}:1]")

    # Detect constant & pure-ID columns
    constant_cols = [c for c in df.columns if df[c].nunique() <= 1]
    id_cols = [c for c in df.columns if df[c].nunique() == n_rows]
    ui.subheader("Constant & Identifier Columns Detected (Auto-Dropped)")
    ui.info(f"Zero-variance columns : {constant_cols}")
    ui.info(f"Pure ID columns       : {id_cols}")

    # Audit legacy cleaned_employee_attrition.csv corruption bug
    corruption_audit: List[Dict[str, Any]] = []
    if legacy_clean_path.exists():
        df_legacy = pd.read_csv(legacy_clean_path)
        common_cols = [c for c in df.columns if c in df_legacy.columns]
        audit_rows = []
        for col in common_cols:
            if len(df_legacy) == len(df):
                diff_cnt = int((df[col] != df_legacy[col]).sum())
                if diff_cnt > 0:
                    pct = diff_cnt / len(df) * 100.0
                    corruption_audit.append(
                        {
                            "column": col,
                            "modified_rows": diff_cnt,
                            "modified_pct": round(pct, 2),
                            "raw_min": float(df[col].min()),
                            "raw_max": float(df[col].max()),
                            "legacy_min": float(df_legacy[col].min()),
                            "legacy_max": float(df_legacy[col].max()),
                        }
                    )
                    audit_rows.append(
                        [
                            col,
                            f"{diff_cnt}",
                            f"{pct:.1f}%",
                            f"[{df[col].min()}, {df[col].max()}]",
                            f"[{df_legacy[col].min()}, {df_legacy[col].max()}]",
                        ]
                    )
        if audit_rows:
            ui.subheader("Legacy Pipeline Audit: Silent Median-Overwrite Corruption Found & Fixed")
            ui.warning(
                f"Original EDA.ipynb modified {len(audit_rows)} numeric columns in-place via IQR median overwrite."
            )
            ui.table(
                ["Column", "Rows Corrupted", "% Affected", "True Raw Range", "Legacy Corrupted Range"],
                audit_rows,
                ["<", ">", ">", ">", ">"],
            )
            ui.success("Training strictly from raw uncorrupted `employee_attrition.csv` + Yeo-Johnson scaling.")

    # Categorical analysis (Chi-Square + Cramer's V)
    ui.subheader("Categorical Feature Association with Attrition (Chi-Square & Cramer's V)")
    cat_cols = [
        c
        for c in df.select_dtypes(include=["object", "string", "category"]).columns
        if c not in ("Attrition", "Over18")
    ]
    cat_summary = []
    cat_table_rows = []
    for col in cat_cols:
        ct = pd.crosstab(df[col], y)
        chi2, p_val, _, _ = stats.chi2_contingency(ct)
        cramers_v = float(np.sqrt(chi2 / (len(df) * max(min(ct.shape) - 1, 1))))
        grp = (
            df.groupby(col)["Attrition"]
            .apply(lambda s: (s == "Yes").mean() * 100.0)
            .sort_values(ascending=False)
        )
        top_cat, top_rate = str(grp.index[0]), float(grp.iloc[0])
        low_cat, low_rate = str(grp.index[-1]), float(grp.iloc[-1])
        cat_summary.append(
            {
                "feature": col,
                "chi2": round(float(chi2), 3),
                "p_value": float(p_val),
                "cramers_v": round(cramers_v, 4),
                "highest_risk_group": f"{top_cat} ({top_rate:.1f}%)",
                "lowest_risk_group": f"{low_cat} ({low_rate:.1f}%)",
            }
        )
    cat_summary.sort(key=lambda x: x["cramers_v"], reverse=True)
    for item in cat_summary:
        cat_table_rows.append(
            [
                item["feature"],
                f"{item['chi2']:.2f}",
                f"{item['p_value']:.2e}",
                f"{item['cramers_v']:.3f}",
                item["highest_risk_group"],
                item["lowest_risk_group"],
            ]
        )
    ui.table(
        ["Feature", "Chi2", "p-value", "Cramer's V", "Highest-Risk Group", "Lowest-Risk Group"],
        cat_table_rows,
        ["<", ">", ">", ">", "<", "<"],
    )

    # Numeric analysis (Skew, IQR Outliers, Point-Biserial r, Mann-Whitney U)
    ui.subheader("Top Numeric Features Ranked by Mann-Whitney U Significance")
    num_cols = [
        c
        for c in df.select_dtypes(include=[np.number]).columns
        if c not in ("EmployeeCount", "StandardHours", "EmployeeNumber")
    ]
    num_summary = []
    for col in num_cols:
        g0 = df.loc[y == 0, col]
        g1 = df.loc[y == 1, col]
        r_pb, _ = stats.pointbiserialr(y, df[col])
        _, p_mw = stats.mannwhitneyu(g1, g0)
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        n_out = int(((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum())
        num_summary.append(
            {
                "feature": col,
                "skew": round(float(df[col].skew()), 2),
                "iqr_outliers": n_out,
                "mean_stay": round(float(g0.mean()), 2),
                "mean_leave": round(float(g1.mean()), 2),
                "point_biserial_r": round(float(r_pb), 4),
                "mw_p_value": float(p_mw),
            }
        )
    num_summary.sort(key=lambda x: x["mw_p_value"])
    num_rows = [
        [
            item["feature"],
            f"{item['skew']:+.2f}",
            f"{item['iqr_outliers']}",
            f"{item['mean_stay']:.2f}",
            f"{item['mean_leave']:.2f}",
            f"{item['point_biserial_r']:+.3f}",
            f"{item['mw_p_value']:.2e}",
        ]
        for item in num_summary[:14]
    ]
    ui.table(
        ["Feature", "Skew", "IQR Out", "Mean(No)", "Mean(Yes)", "Corr(r)", "MW p-value"],
        num_rows,
        ["<", ">", ">", ">", ">", ">", ">"],
    )

    # Multicollinearity & High-Risk Interactions
    corr_mat = df[num_cols].corr()
    collinear_pairs = []
    for i in range(len(num_cols)):
        for j in range(i + 1, len(num_cols)):
            val = float(corr_mat.iloc[i, j])
            if abs(val) >= 0.60:
                collinear_pairs.append((num_cols[i], num_cols[j], val))
    collinear_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

    ui.subheader("Multicollinearity Audit (|r| >= 0.60) & High-Risk Interaction Discovery")
    for a, b, val in collinear_pairs[:6]:
        ui.info(f"Collinear Pair : {a:25s} <-> {b:25s} (r = {val:+.3f})")

    inter_rows = []
    for label, mask in [
        ("OverTime=Yes & MaritalStatus=Single", (df["OverTime"] == "Yes") & (df["MaritalStatus"] == "Single")),
        ("OverTime=Yes & StockOptionLevel=0", (df["OverTime"] == "Yes") & (df["StockOptionLevel"] == 0)),
        ("OverTime=Yes & Travel_Frequently", (df["OverTime"] == "Yes") & (df["BusinessTravel"] == "Travel_Frequently")),
        ("JobRole=Sales Representative", df["JobRole"] == "Sales Representative"),
        ("Early Tenure (YearsAtCompany <= 1)", df["YearsAtCompany"] <= 1),
        ("Poor WorkLifeBalance (WLB == 1)", df["WorkLifeBalance"] == 1),
        ("Baseline Overall Dataset Average", pd.Series(True, index=df.index)),
    ]:
        cnt = int(mask.sum())
        rate = float(y[mask].mean() * 100.0)
        lift = rate / pos_rate
        inter_rows.append([label, f"{cnt}", f"{rate:.1f}%", f"{lift:.2f}x"])
    print()
    ui.table(["High-Risk Subgroup", "Count", "Attrition Rate", "Risk Lift"], inter_rows, ["<", ">", ">", ">"])

    return {
        "shape": {"rows": n_rows, "columns": n_cols},
        "missing_values": n_missing,
        "duplicates": n_dups,
        "class_balance": {"no": n_neg, "yes": n_pos, "yes_pct": round(pos_rate, 2), "ratio": round(imbalance_ratio, 3)},
        "constant_columns": constant_cols,
        "id_columns": id_cols,
        "legacy_corruption_audit": corruption_audit,
        "categorical_associations": cat_summary,
        "numeric_associations": num_summary,
        "high_collinearity_pairs": [{"a": a, "b": b, "r": round(r, 4)} for a, b, r in collinear_pairs],
    }

## Helper 3: Evaluation Metrics, Threshold Tuning & Leak-Free Fold Cache

In [ ]:
# =============================================================================
# EVALUATION & THRESHOLD HELPERS
# =============================================================================
def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, Any]:
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = (int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1]))
    y_prob_clip = np.clip(y_prob, 1e-6, 1.0 - 1e-6)
    return {
        "threshold": round(float(threshold), 4),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "f1_5": float(fbeta_score(y_true, y_pred, beta=1.5, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "log_loss": float(log_loss(y_true, y_prob_clip)),
        "brier": float(brier_score_loss(y_true, y_prob_clip)),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def optimize_threshold_oof(y_true: np.ndarray, oof_prob: np.ndarray, beta: float = 1.0) -> Tuple[float, float]:
    """Find threshold maximizing F-beta strictly on Out-Of-Fold CV probabilities."""
    thresholds = np.linspace(0.08, 0.85, 155)
    best_thr, best_score = 0.5, -1.0
    for thr in thresholds:
        pred = (oof_prob >= thr).astype(int)
        score = float(fbeta_score(y_true, pred, beta=beta, zero_division=0))
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr, best_score


def build_leak_free_fold_cache(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    preprocessor_template: Pipeline,
    n_splits: int,
    seed: int,
    ui: TerminalUI,
    label: str,
) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """
    Fit a fresh clone of `preprocessor_template` strictly on each training fold
    and transform that fold's validation split. Mathematically identical to
    fitting a sklearn Pipeline inside CV (zero leakage!), while avoiding
    redundant Yeo-Johnson PowerTransformer refits across hundreds of HPO trials.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    cache = []
    t0 = time.perf_counter()
    for tr_idx, val_idx in skf.split(X_train, y_train):
        pre = clone(preprocessor_template)
        X_f_tr = pre.fit_transform(X_train.iloc[tr_idx], y_train[tr_idx])
        X_f_val = pre.transform(X_train.iloc[val_idx])
        cache.append((X_f_tr, y_train[tr_idx], X_f_val, y_train[val_idx], val_idx))
    dt = time.perf_counter() - t0
    ui.success(
        f"Built leak-free {n_splits}-fold preprocessed cache for {label} "
        f"(transformed shape={cache[0][0].shape[1]} features, {dt:.2f}s)"
    )
    return cache

## Helper 4: Live Parallel-Fold HPO & OOF Cross-Validation Engines

In [ ]:
# =============================================================================
# STAGE 3: LIVE TERMINAL-VISIBLE HPO ENGINE (WITH PARALLEL FOLDS)
# =============================================================================
def _fit_and_score_fold(
    base_estimator: BaseEstimator,
    params: Dict[str, Any],
    X_f_tr: np.ndarray,
    y_f_tr: np.ndarray,
    X_f_val: np.ndarray,
    y_f_val: np.ndarray,
) -> Tuple[float, float, float]:
    est = clone(base_estimator)
    est.set_params(**params)
    est.fit(X_f_tr, y_f_tr)
    probs = est.predict_proba(X_f_val)[:, 1]
    probs_clip = np.clip(probs, 1e-6, 1.0 - 1e-6)
    return (
        float(roc_auc_score(y_f_val, probs)),
        float(average_precision_score(y_f_val, probs)),
        float(log_loss(y_f_val, probs_clip)),
    )


def run_live_hpo(
    model_name: str,
    base_estimator: BaseEstimator,
    param_distributions: Dict[str, Any],
    fold_cache: List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]],
    n_iter: int,
    seed: int,
    ui: TerminalUI,
    param_validator: Optional[Any] = None,
) -> Tuple[BaseEstimator, Dict[str, Any], Dict[str, float]]:
    """
    Stratified K-Fold Hyperparameter Optimization with parallel fold execution
    and live terminal progress bar / telemetry.
    Optimizes composite generalization objective:
        0.55 * ROC-AUC + 0.35 * PR-AUC - 0.10 * LogLoss
    """
    n_folds = len(fold_cache)
    ui.subheader(f"HPO Tuning: {model_name} ({n_iter} trials x {n_folds} folds = {n_iter * n_folds} fits)")

    sampler = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=seed))
    best_obj = -1e9
    best_params: Dict[str, Any] = {}
    best_summary: Dict[str, float] = {}

    t_start = time.perf_counter()
    print_every = max(1, n_iter // 8)
    n_fold_workers = min(n_folds, os.cpu_count() or 1)

    for trial_idx, raw_params in enumerate(sampler, start=1):
        params = param_validator(raw_params) if param_validator else dict(raw_params)

        fold_results = Parallel(n_jobs=n_fold_workers, prefer="threads")(
            delayed(_fit_and_score_fold)(base_estimator, params, X_f_tr, y_f_tr, X_f_val, y_f_val)
            for X_f_tr, y_f_tr, X_f_val, y_f_val, _ in fold_cache
        )
        fold_aucs = [r[0] for r in fold_results]
        fold_prs = [r[1] for r in fold_results]
        fold_lls = [r[2] for r in fold_results]

        mean_auc = float(np.mean(fold_aucs))
        std_auc = float(np.std(fold_aucs))
        mean_pr = float(np.mean(fold_prs))
        mean_ll = float(np.mean(fold_lls))
        composite_obj = 0.55 * mean_auc + 0.35 * mean_pr - 0.10 * mean_ll

        is_best = composite_obj > best_obj
        if is_best:
            best_obj = composite_obj
            best_params = dict(params)
            best_summary = {
                "cv_roc_auc": mean_auc,
                "cv_roc_auc_std": std_auc,
                "cv_pr_auc": mean_pr,
                "cv_log_loss": mean_ll,
                "composite_score": composite_obj,
            }

        elapsed = time.perf_counter() - t_start
        eta = (elapsed / trial_idx) * (n_iter - trial_idx)
        bar = ui.progress_bar(trial_idx, n_iter, width=20)

        if is_best or (trial_idx % print_every == 0) or (trial_idx == n_iter):
            badge = f"{ui.GREEN}{ui.BOLD}{ui.STAR} NEW BEST{ui.RESET}" if is_best else f"{ui.DIM}trial   {ui.RESET}"
            print(
                f"   {bar} {trial_idx:2d}/{n_iter:2d} | {badge} | "
                f"AUC={mean_auc:.4f} (best={best_summary['cv_roc_auc']:.4f}) | "
                f"PR={mean_pr:.4f} | LL={mean_ll:.4f} | ETA={eta:4.1f}s",
                flush=True,
            )

    best_est = clone(base_estimator)
    best_est.set_params(**best_params)

    clean_p = {
        k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else (int(v) if isinstance(v, np.integer) else v))
        for k, v in best_params.items()
    }
    ui.success(
        f"{model_name} HPO Done ({time.perf_counter() - t_start:.1f}s) -> "
        f"Best CV ROC-AUC={best_summary['cv_roc_auc']:.4f} +/- {best_summary['cv_roc_auc_std']:.4f} | "
        f"PR-AUC={best_summary['cv_pr_auc']:.4f} | LogLoss={best_summary['cv_log_loss']:.4f}"
    )
    ui.info(f"Best Params: {clean_p}")
    return best_est, clean_p, best_summary


def evaluate_oof_from_cache(
    model_name: str,
    estimator: BaseEstimator,
    fold_cache: List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]],
    n_samples: int,
    ui: TerminalUI,
    parallel_folds: bool = True,
) -> Tuple[np.ndarray, Dict[str, float]]:
    """Generate leak-free out-of-fold probabilities across the CV fold cache."""
    t0 = time.perf_counter()

    def _run_oof_fold(X_f_tr, y_f_tr, X_f_val, y_f_val, val_idx):
        est = clone(estimator)
        est.fit(X_f_tr, y_f_tr)
        p = est.predict_proba(X_f_val)[:, 1]
        return val_idx, y_f_val, p, float(roc_auc_score(y_f_val, p))

    if parallel_folds:
        n_workers = min(len(fold_cache), os.cpu_count() or 1)
        fold_outs = Parallel(n_jobs=n_workers, prefer="threads")(
            delayed(_run_oof_fold)(X_f_tr, y_f_tr, X_f_val, y_f_val, val_idx)
            for X_f_tr, y_f_tr, X_f_val, y_f_val, val_idx in fold_cache
        )
    else:
        fold_outs = []
        n_folds = len(fold_cache)
        for f_idx, (X_f_tr, y_f_tr, X_f_val, y_f_val, val_idx) in enumerate(fold_cache, start=1):
            out = _run_oof_fold(X_f_tr, y_f_tr, X_f_val, y_f_val, val_idx)
            fold_outs.append(out)
            bar = ui.progress_bar(f_idx, n_folds, width=16)
            print(
                f"      {model_name:28s} {bar} Fold {f_idx:2d}/{n_folds:2d} | Fold ROC-AUC={out[3]:.4f}",
                flush=True,
            )

    oof_probs = np.zeros(n_samples, dtype=float)
    y_all = np.zeros(n_samples, dtype=int)
    fold_aucs: List[float] = []
    for val_idx, y_f_val, p, f_auc in fold_outs:
        oof_probs[val_idx] = p
        y_all[val_idx] = y_f_val
        fold_aucs.append(f_auc)

    dt = time.perf_counter() - t0
    stats_out = {
        "oof_roc_auc": float(roc_auc_score(y_all, oof_probs)),
        "fold_roc_auc_mean": float(np.mean(fold_aucs)),
        "fold_roc_auc_std": float(np.std(fold_aucs)),
        "oof_pr_auc": float(average_precision_score(y_all, oof_probs)),
        "oof_log_loss": float(log_loss(y_all, np.clip(oof_probs, 1e-6, 1.0 - 1e-6))),
        "cv_seconds": round(dt, 2),
    }
    ui.info(
        f"{model_name:28s} | {len(fold_cache)}-Fold OOF ROC-AUC={stats_out['oof_roc_auc']:.4f} "
        f"(fold {stats_out['fold_roc_auc_mean']:.4f}+/-{stats_out['fold_roc_auc_std']:.4f}) | "
        f"PR-AUC={stats_out['oof_pr_auc']:.4f} | LogLoss={stats_out['oof_log_loss']:.4f} ({dt:.1f}s)"
    )
    return oof_probs, stats_out

## Helper 5: Comprehensive 11-Plot Feature EDA & Model Comparison Suite

In [ ]:
# =============================================================================
# STAGE 7: COMPREHENSIVE 11-PLOT FEATURE EDA & MODEL COMPARISON VISUALIZATION
# =============================================================================
def generate_all_plots(
    df_raw: pd.DataFrame,
    df_engineered_all: pd.DataFrame,
    y_train: np.ndarray,
    y_test: np.ndarray,
    oof_probs: Dict[str, np.ndarray],
    test_probs: Dict[str, np.ndarray],
    test_metrics_f1: Dict[str, Dict[str, Any]],
    test_metrics_hr: Dict[str, Dict[str, Any]],
    train_metrics_f1: Dict[str, Dict[str, Any]],
    oof_stats: Dict[str, Dict[str, float]],
    corr_oof: pd.DataFrame,
    soft_vote_weights: Dict[str, float],
    stacking_meta_coefs: Dict[str, float],
    legacy_ref: Dict[str, Dict[str, float]],
    subgroup_report: Dict[str, List[Dict[str, Any]]],
    champion_name: str,
    opt_thr_f1: Dict[str, float],
    opt_thr_hr: Dict[str, float],
    fitted_models: Dict[str, BaseEstimator],
    feature_names: List[str],
    plots_dir: Path,
    show_plots: bool,
    ui: TerminalUI,
) -> List[Tuple[str, str]]:
    """Generate 11 comprehensive feature EDA and model comparison plots."""
    plots_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", font_scale=1.0)
    saved_catalog: List[Tuple[str, str]] = []
    y_all = (df_raw["Attrition"] == "Yes").astype(int)

    def _save_fig(fig: plt.Figure, filename: str, desc: str) -> None:
        out_path = plots_dir / filename
        fig.savefig(out_path, dpi=160, bbox_inches="tight")
        saved_catalog.append((str(out_path), desc))
        if show_plots:
            plt.show()
        plt.close(fig)

    # -------------------------------------------------------------------------
    # PLOT 01: EDA — Target Distribution & Demographic / Organizational Drivers
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(16, 11.5))

    counts = df_raw["Attrition"].value_counts()
    bars = axes[0, 0].bar(counts.index, counts.values, color=["#2E86AB", "#E63946"], edgecolor="black", alpha=0.9)
    for bar in bars:
        h = bar.get_height()
        axes[0, 0].text(
            bar.get_x() + bar.get_width() / 2, h + 18,
            f"{int(h)} ({h / len(df_raw) * 100:.2f}%)",
            ha="center", va="bottom", fontweight="bold",
        )
    axes[0, 0].set_title("Target Distribution (Class Imbalance 5.20:1)", fontweight="bold")
    axes[0, 0].set_ylabel("Employee Count")
    axes[0, 0].set_ylim(0, max(counts.values) * 1.15)

    dept_grp = df_raw.groupby("Department")["Attrition"].agg(
        count="count", rate=lambda s: (s == "Yes").mean() * 100.0
    ).sort_values("rate", ascending=False)
    bars_d = axes[0, 1].bar(dept_grp.index, dept_grp["rate"], color=["#E63946", "#F4A261", "#457B9D"], edgecolor="black")
    for bar, (_, row) in zip(bars_d, dept_grp.iterrows()):
        h = bar.get_height()
        axes[0, 1].text(
            bar.get_x() + bar.get_width() / 2, h + 0.5,
            f"{h:.1f}%\n(n={int(row['count'])})", ha="center", va="bottom", fontweight="bold", fontsize=9,
        )
    axes[0, 1].set_title("Attrition Rate (%) by Department", fontweight="bold")
    axes[0, 1].set_ylabel("Attrition Rate (%)")
    axes[0, 1].set_ylim(0, 27)

    role_grp = df_raw.groupby("JobRole")["Attrition"].agg(
        count="count", rate=lambda s: (s == "Yes").mean() * 100.0
    ).sort_values("rate", ascending=True)
    role_cols = ["#E63946" if v >= 22 else ("#F4A261" if v >= 15 else "#457B9D") for v in role_grp["rate"]]
    axes[1, 0].barh(role_grp.index, role_grp["rate"], color=role_cols, edgecolor="black", alpha=0.9)
    for idx, (_, row) in enumerate(role_grp.iterrows()):
        axes[1, 0].text(row["rate"] + 0.6, idx, f"{row['rate']:.1f}% (n={int(row['count'])})", va="center", fontsize=8.8, fontweight="bold")
    axes[1, 0].set_title("Attrition Rate (%) by Job Role (Restored High-Signal Feature)", fontweight="bold")
    axes[1, 0].set_xlabel("Attrition Rate (%)")
    axes[1, 0].set_xlim(0, 52)

    age_bins = pd.cut(df_raw["Age"], bins=[17, 25, 32, 40, 50, 65], labels=["18-25", "26-32", "33-40", "41-50", "51-60"])
    age_grp = df_raw.assign(AttrNum=y_all, AgeBin=age_bins).groupby("AgeBin", observed=True)["AttrNum"].agg(count="count", rate=lambda s: s.mean() * 100.0)
    bars_a = axes[1, 1].bar(age_grp.index.astype(str), age_grp["rate"], color=["#E63946", "#F4A261", "#457B9D", "#2A9D8F", "#264653"], edgecolor="black")
    for bar, (_, row) in zip(bars_a, age_grp.iterrows()):
        h = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width() / 2, h + 0.7, f"{h:.1f}%\n(n={int(row['count'])})", ha="center", va="bottom", fontweight="bold", fontsize=9)
    axes[1, 1].set_title("Attrition Rate (%) by Age Cohort", fontweight="bold")
    axes[1, 1].set_ylabel("Attrition Rate (%)")
    axes[1, 1].set_ylim(0, 46)

    fig.suptitle("01 | From-Scratch EDA: Target Balance, Department, Job Role & Age", fontsize=15, fontweight="bold", y=0.995)
    fig.tight_layout()
    _save_fig(fig, "01_eda_target_and_demographics.png", "Target imbalance, Department, JobRole & Age EDA")

    # -------------------------------------------------------------------------
    # PLOT 02: EDA — Compensation, Satisfaction, Work-Life Balance & Tenure
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(16, 11.5))

    inc_bins = pd.cut(
        df_raw["MonthlyIncome"],
        bins=[0, 3000, 6000, 9000, 12000, 15000, df_raw["MonthlyIncome"].max() + 1],
        labels=["0-3k", "3k-6k", "6k-9k", "9k-12k", "12k-15k", "15k+"],
    )
    inc_grp = df_raw.assign(AttrNum=y_all, IncBin=inc_bins).groupby("IncBin", observed=True)["AttrNum"].agg(count="count", rate=lambda s: s.mean() * 100.0)
    bars_i = axes[0, 0].bar(inc_grp.index.astype(str), inc_grp["rate"], color="#E63946", alpha=0.85, edgecolor="black")
    for bar, (_, row) in zip(bars_i, inc_grp.iterrows()):
        h = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width() / 2, h + 0.6, f"{h:.1f}%\n(n={int(row['count'])})", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
    axes[0, 0].set_title("Attrition Rate (%) by Monthly Income Bracket", fontweight="bold")
    axes[0, 0].set_ylabel("Attrition Rate (%)")
    axes[0, 0].set_ylim(0, 36)

    sns.boxplot(
        data=df_raw, x="Attrition", y="MonthlyIncome", hue="Attrition",
        palette={"No": "#2E86AB", "Yes": "#E63946"}, legend=False, ax=axes[0, 1],
    )
    axes[0, 1].set_title("Uncorrupted Monthly Income Distribution (Stay vs Leave)", fontweight="bold")
    axes[0, 1].set_ylabel("Monthly Income ($)")

    sat_melt = []
    for feat in ["WorkLifeBalance", "JobSatisfaction", "EnvironmentSatisfaction", "JobInvolvement"]:
        rates = df_raw.assign(AttrNum=y_all).groupby(feat)["AttrNum"].mean() * 100.0
        for lvl, r in rates.items():
            sat_melt.append({"Dimension": feat.replace("Satisfaction", "Sat."), "Rating": str(lvl), "Rate": r})
    sat_df = pd.DataFrame(sat_melt)
    sns.barplot(data=sat_df, x="Dimension", y="Rate", hue="Rating", palette="RdYlGn", ax=axes[1, 0])
    axes[1, 0].set_title("Attrition Rate (%) Across Satisfaction & Engagement Ratings (1=Low, 4=High)", fontweight="bold")
    axes[1, 0].set_ylabel("Attrition Rate (%)")
    axes[1, 0].legend(title="Rating", ncol=4, loc="upper right", fontsize=8.5)

    ten_bins = pd.cut(df_raw["YearsAtCompany"], bins=[-1, 1, 3, 5, 10, 45], labels=["0-1 yr", "2-3 yr", "4-5 yr", "6-10 yr", "11+ yr"])
    ten_grp = df_raw.assign(AttrNum=y_all, TB=ten_bins).groupby("TB", observed=True)["AttrNum"].agg(count="count", rate=lambda s: s.mean() * 100.0)
    bars_t = axes[1, 1].bar(ten_grp.index.astype(str), ten_grp["rate"], color=["#E63946", "#F4A261", "#457B9D", "#2A9D8F", "#264653"], edgecolor="black")
    for bar, (_, row) in zip(bars_t, ten_grp.iterrows()):
        h = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width() / 2, h + 0.8, f"{h:.1f}%\n(n={int(row['count'])})", ha="center", va="bottom", fontweight="bold", fontsize=9)
    axes[1, 1].set_title("Early-Tenure Attrition Cliff (Years at Company)", fontweight="bold")
    axes[1, 1].set_ylabel("Attrition Rate (%)")
    axes[1, 1].set_ylim(0, 43)

    fig.suptitle("02 | From-Scratch EDA: Compensation, Satisfaction Ratings & Tenure Cliff", fontsize=15, fontweight="bold", y=0.995)
    fig.tight_layout()
    _save_fig(fig, "02_eda_income_tenure_satisfaction.png", "Income brackets, boxplots, satisfaction & tenure cliff")

    # -------------------------------------------------------------------------
    # PLOT 03: EDA — Interaction Heatmaps & Multicollinearity Matrix
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    piv1 = df_raw.assign(AttrNum=y_all).pivot_table(index="MaritalStatus", columns="OverTime", values="AttrNum", aggfunc="mean") * 100.0
    sns.heatmap(piv1, annot=True, fmt=".1f", cmap="YlOrRd", ax=axes[0, 0], cbar_kws={"label": "Attrition %"})
    axes[0, 0].set_title("Interaction: OverTime x MaritalStatus (%)", fontweight="bold")

    piv2 = df_raw.assign(AttrNum=y_all).pivot_table(index="StockOptionLevel", columns="OverTime", values="AttrNum", aggfunc="mean") * 100.0
    sns.heatmap(piv2, annot=True, fmt=".1f", cmap="RdYlBu_r", ax=axes[0, 1], cbar_kws={"label": "Attrition %"})
    axes[0, 1].set_title("Non-Linear U-Shape: OverTime x StockOptionLevel (%)", fontweight="bold")

    piv3 = df_raw.assign(AttrNum=y_all).pivot_table(index="BusinessTravel", columns="OverTime", values="AttrNum", aggfunc="mean") * 100.0
    sns.heatmap(piv3, annot=True, fmt=".1f", cmap="YlOrRd", ax=axes[1, 0], cbar_kws={"label": "Attrition %"})
    axes[1, 0].set_title("Interaction: OverTime x BusinessTravel (%)", fontweight="bold")

    corr_cols = [
        "Age", "MonthlyIncome", "JobLevel", "TotalWorkingYears", "YearsAtCompany",
        "YearsInCurrentRole", "YearsSinceLastPromotion", "YearsWithCurrManager",
        "PercentSalaryHike", "PerformanceRating", "NumCompaniesWorked", "DistanceFromHome",
    ]
    sns.heatmap(df_raw[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1, 1], annot_kws={"size": 8})
    axes[1, 1].set_title("Numeric Feature Correlation Matrix (Multicollinearity Clusters)", fontweight="bold")

    fig.suptitle("03 | Deep EDA: High-Risk Feature Interactions & Collinearity Audit", fontsize=15, fontweight="bold", y=0.995)
    fig.tight_layout()
    _save_fig(fig, "03_eda_interactions_and_correlations.png", "OverTime interaction heatmaps & feature correlation matrix")

    # -------------------------------------------------------------------------
    # PLOT 04: EDA — Continuous Numeric Feature Distributions (Stay vs Leave)
    # -------------------------------------------------------------------------
    top_num_feats = [
        "Age", "MonthlyIncome", "TotalWorkingYears",
        "YearsAtCompany", "YearsInCurrentRole", "YearsWithCurrManager",
        "DistanceFromHome", "NumCompaniesWorked", "JobLevel",
    ]
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    for ax, feat in zip(axes.flat, top_num_feats):
        sns.kdeplot(
            data=df_raw, x=feat, hue="Attrition", fill=True, common_norm=False,
            palette={"No": "#2E86AB", "Yes": "#E63946"}, alpha=0.45, ax=ax,
        )
        m_no = df_raw.loc[df_raw["Attrition"] == "No", feat].mean()
        m_yes = df_raw.loc[df_raw["Attrition"] == "Yes", feat].mean()
        ax.axvline(m_no, color="#2E86AB", linestyle="--", linewidth=1.2)
        ax.axvline(m_yes, color="#E63946", linestyle="--", linewidth=1.2)
        ax.set_title(f"{feat} (Mean No={m_no:.1f} vs Yes={m_yes:.1f})", fontweight="bold", fontsize=10)
    fig.suptitle("04 | Feature Distributions: Top 9 Numeric Predictors (Stay vs Leave)", fontsize=15, fontweight="bold", y=0.995)
    fig.tight_layout()
    _save_fig(fig, "04_eda_feature_distributions.png", "3x3 density plots of top 9 numeric features by Attrition")

    # -------------------------------------------------------------------------
    # PLOT 05: Engineered Domain Features Discriminative Power
    # -------------------------------------------------------------------------
    eng_plot_df = df_engineered_all.copy()
    eng_plot_df["Attrition"] = df_raw["Attrition"].values
    eng_feats = [
        ("BurnoutIndex", "Composite Burnout Index"),
        ("EngagementScore", "Engagement & Satisfaction Score"),
        ("IncomeVsRoleMedian", "Income Relative to JobRole Median"),
        ("TenureRatio", "Tenure / Total Experience Ratio"),
        ("PromotionStagnation", "Promotion Stagnation Ratio"),
        ("LowSatisfactionCount", "Count of Lowest (Rating=1) Scores"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(16.5, 10))
    for ax, (col, title) in zip(axes.flat, eng_feats):
        if col == "LowSatisfactionCount":
            grp = eng_plot_df.assign(AttrNum=y_all).groupby(col)["AttrNum"].mean() * 100.0
            ax.bar(grp.index.astype(int).astype(str), grp.values, color="#E76F51", edgecolor="black", alpha=0.9)
            for idx, val in enumerate(grp.values):
                ax.text(idx, val + 0.8, f"{val:.1f}%", ha="center", fontweight="bold", fontsize=9)
            ax.set_ylabel("Attrition Rate (%)")
            ax.set_xlabel("Number of Rating=1 Dimensions")
        else:
            sns.kdeplot(
                data=eng_plot_df, x=col, hue="Attrition", fill=True, common_norm=False,
                palette={"No": "#2E86AB", "Yes": "#E63946"}, alpha=0.45, ax=ax,
            )
        ax.set_title(title, fontweight="bold", fontsize=10.5)
    fig.suptitle("05 | Feature Engineering Validation: Newly Engineered Domain Signals", fontsize=15, fontweight="bold", y=0.995)
    fig.tight_layout()
    _save_fig(fig, "05_eda_engineered_features.png", "Distributions of newly engineered HR domain features")

    # -------------------------------------------------------------------------
    # PLOT 06: Multi-Model Feature Importance Comparison (2x2)
    # -------------------------------------------------------------------------
    clean_names = [n.replace("num__", "").replace("cat__", "") for n in feature_names]
    fig, axes = plt.subplots(2, 2, figsize=(17, 13))

    norm_imps: Dict[str, np.ndarray] = {}

    # 6A: Logistic Regression Signed Standardized Coefficients
    if "Logistic Regression" in fitted_models:
        coefs = fitted_models["Logistic Regression"].coef_[0]
        norm_imps["LR"] = np.abs(coefs) / (np.abs(coefs).sum() + 1e-12)
        lr_df = pd.DataFrame({"Feature": clean_names, "Coef": coefs, "Abs": np.abs(coefs)})
        top_lr = lr_df.sort_values("Abs", ascending=False).head(15).sort_values("Coef")
        cols_lr = ["#E63946" if v > 0 else "#2E86AB" for v in top_lr["Coef"]]
        axes[0, 0].barh(top_lr["Feature"], top_lr["Coef"], color=cols_lr, edgecolor="black", alpha=0.9)
        axes[0, 0].axvline(0, color="black", linewidth=0.9)
        axes[0, 0].set_title("Logistic Regression: Top 15 Signed Coefficients\n(Red = Increases Risk | Blue = Retention Protective)", fontweight="bold")
        axes[0, 0].set_xlabel("Standardized Log-Odds Coefficient")

    # 6B: Random Forest Feature Importances
    if "Random Forest" in fitted_models:
        rf_imp = fitted_models["Random Forest"].feature_importances_
        norm_imps["RF"] = rf_imp / (rf_imp.sum() + 1e-12)
        rf_df = pd.DataFrame({"Feature": clean_names, "Imp": rf_imp}).sort_values("Imp", ascending=False).head(15).iloc[::-1]
        axes[0, 1].barh(rf_df["Feature"], rf_df["Imp"], color="#2A9D8F", edgecolor="black", alpha=0.9)
        axes[0, 1].set_title("Random Forest: Top 15 Feature Importances", fontweight="bold")
        axes[0, 1].set_xlabel("Impurity Decrease Importance")

    # 6C: XGBoost Feature Importances
    if "XGBoost" in fitted_models:
        xgb_imp = fitted_models["XGBoost"].feature_importances_
        norm_imps["XGB"] = xgb_imp / (xgb_imp.sum() + 1e-12)
        xgb_df = pd.DataFrame({"Feature": clean_names, "Imp": xgb_imp}).sort_values("Imp", ascending=False).head(15).iloc[::-1]
        axes[1, 0].barh(xgb_df["Feature"], xgb_df["Imp"], color="#F4A261", edgecolor="black", alpha=0.9)
        axes[1, 0].set_title("XGBoost: Top 15 Feature Importances", fontweight="bold")
        axes[1, 0].set_xlabel("Normalized Gain Importance")

    # 6D: Cross-Model Consensus Importance (LR + RF + XGB + Extra Trees + GB)
    for extra_key, short_k in [("Extra Trees", "ET"), ("Gradient Boosting", "GB")]:
        if extra_key in fitted_models and hasattr(fitted_models[extra_key], "feature_importances_"):
            v = fitted_models[extra_key].feature_importances_
            norm_imps[short_k] = v / (v.sum() + 1e-12)
    consensus = np.mean(list(norm_imps.values()), axis=0)
    cons_df = pd.DataFrame({"Feature": clean_names, "Consensus": consensus}).sort_values("Consensus", ascending=False).head(15).iloc[::-1]
    axes[1, 1].barh(cons_df["Feature"], cons_df["Consensus"] * 100.0, color="#6A4C93", edgecolor="black", alpha=0.9)
    for idx, val in enumerate(cons_df["Consensus"] * 100.0):
        axes[1, 1].text(val + 0.08, idx, f"{val:.2f}%", va="center", fontsize=8.5, fontweight="bold")
    axes[1, 1].set_title("Cross-Model Consensus Top 15 Features (Mean Across 5 Models)", fontweight="bold")
    axes[1, 1].set_xlabel("Average Normalized Importance Share (%)")

    fig.suptitle("06 | Feature Importance Comparison Across Linear, Bagging & Boosting Models", fontsize=15, fontweight="bold", y=0.995)
    fig.tight_layout()
    _save_fig(fig, "06_feature_importance_multi_model.png", "LR, RF, XGB & 5-model consensus feature importance charts")

    # -------------------------------------------------------------------------
    # PLOT 07: Comprehensive Model Comparison — Metrics, Generalization & Baseline Lift
    # -------------------------------------------------------------------------
    model_order = list(test_metrics_f1.keys())
    fig, axes = plt.subplots(2, 2, figsize=(17.5, 12.5))
    x = np.arange(len(model_order))
    w = 0.25

    # 7A: OOF ROC-AUC vs Test ROC-AUC vs Test PR-AUC
    cv_aucs = [oof_stats[m]["oof_roc_auc"] for m in model_order]
    te_aucs = [test_metrics_f1[m]["roc_auc"] for m in model_order]
    te_prs = [test_metrics_f1[m]["pr_auc"] for m in model_order]
    axes[0, 0].bar(x - w, cv_aucs, w, label="10-Fold OOF ROC-AUC", color="#1D3557")
    axes[0, 0].bar(x, te_aucs, w, label="Holdout Test ROC-AUC", color="#457B9D")
    axes[0, 0].bar(x + w, te_prs, w, label="Holdout Test PR-AUC", color="#2A9D8F")
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(model_order, rotation=30, ha="right", fontsize=8.8)
    axes[0, 0].set_ylim(0, 1.04)
    axes[0, 0].set_ylabel("Score")
    axes[0, 0].set_title("Ranking Quality: 10-Fold OOF AUC vs Holdout Test AUC & PR-AUC", fontweight="bold")
    axes[0, 0].legend(fontsize=8.5)

    # 7B: Classification Metrics (Accuracy, Precision, Recall, F1)
    w4 = 0.20
    axes[0, 1].bar(x - 1.5 * w4, [test_metrics_f1[m]["accuracy"] for m in model_order], w4, label="Accuracy", color="#264653")
    axes[0, 1].bar(x - 0.5 * w4, [test_metrics_f1[m]["precision"] for m in model_order], w4, label="Precision", color="#2A9D8F")
    axes[0, 1].bar(x + 0.5 * w4, [test_metrics_f1[m]["recall"] for m in model_order], w4, label="Recall", color="#F4A261")
    axes[0, 1].bar(x + 1.5 * w4, [test_metrics_f1[m]["f1"] for m in model_order], w4, label="F1-Score", color="#E63946")
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(model_order, rotation=30, ha="right", fontsize=8.8)
    axes[0, 1].set_ylim(0, 1.04)
    axes[0, 1].set_ylabel("Score")
    axes[0, 1].set_title("Holdout Classification Metrics (OOF Balanced-F1 Threshold)", fontweight="bold")
    axes[0, 1].legend(fontsize=8.5, ncol=2)

    # 7C: Overfitting / Generalization Gap (Train AUC - Test AUC)
    gaps = [train_metrics_f1[m]["roc_auc"] - test_metrics_f1[m]["roc_auc"] for m in model_order]
    gap_cols = ["#2A9D8F" if g < 0.08 else ("#F4A261" if g < 0.14 else "#E63946") for g in gaps]
    axes[1, 0].barh(model_order, gaps, color=gap_cols, edgecolor="black", alpha=0.9)
    for idx, g in enumerate(gaps):
        axes[1, 0].text(g + 0.003, idx, f"{g:+.4f}", va="center", fontsize=8.8, fontweight="bold")
    axes[1, 0].axvline(0.08, color="green", linestyle="--", alpha=0.6, label="Healthy Generalization (<0.08)")
    axes[1, 0].set_xlabel("Train ROC-AUC minus Test ROC-AUC (Lower = Less Overfitting)")
    axes[1, 0].set_title("Generalization Audit Across All Models & Ensembles", fontweight="bold")
    axes[1, 0].legend(fontsize=8.5)

    # 7D: Legacy Baseline vs New Upgraded Pipeline Comparison
    cmp_models = ["Logistic Regression", "Random Forest", "XGBoost", "Weighted Soft-Vote Ensemble", "Stacking Ensemble"]
    cmp_x = np.arange(len(cmp_models))
    old_auc = [legacy_ref.get(m, {"roc_auc": 0.8088})["roc_auc"] for m in cmp_models]
    new_auc = [test_metrics_f1[m]["roc_auc"] for m in cmp_models]
    old_f1 = [legacy_ref.get(m, {"f1": 0.4865})["f1"] for m in cmp_models]
    new_f1 = [test_metrics_f1[m]["f1"] for m in cmp_models]
    wb = 0.19
    axes[1, 1].bar(cmp_x - 1.5 * wb, old_auc, wb, label="Legacy ROC-AUC", color="#A8DADC")
    axes[1, 1].bar(cmp_x - 0.5 * wb, new_auc, wb, label="New HPO ROC-AUC", color="#1D3557")
    axes[1, 1].bar(cmp_x + 0.5 * wb, old_f1, wb, label="Legacy F1", color="#F4A261", alpha=0.65)
    axes[1, 1].bar(cmp_x + 1.5 * wb, new_f1, wb, label="New HPO F1", color="#E63946")
    axes[1, 1].set_xticks(cmp_x)
    axes[1, 1].set_xticklabels(cmp_models, rotation=22, ha="right", fontsize=8.8)
    axes[1, 1].set_ylim(0, 1.05)
    axes[1, 1].set_ylabel("Holdout Test Score")
    axes[1, 1].set_title("Before vs After: Legacy Saved Models vs Upgraded HPO + Ensembles", fontweight="bold")
    axes[1, 1].legend(fontsize=8.2, ncol=2)

    fig.suptitle("07 | Full Model Performance Comparison, Generalization & Baseline Lift", fontsize=15, fontweight="bold", y=0.995)
    fig.tight_layout()
    _save_fig(fig, "07_model_comparison_metrics.png", "All 9 models metric bars, overfit gap & legacy-vs-new lift")

    # -------------------------------------------------------------------------
    # PLOT 08: Radar Chart Comparison + Ensemble Diversity & Learned Weights
    # -------------------------------------------------------------------------
    fig = plt.figure(figsize=(18.5, 6.2))

    # 8A: Radar / Spider Comparison
    ax_radar = fig.add_subplot(1, 3, 1, polar=True)
    radar_cats = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC", "PR-AUC"]
    radar_keys = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
    angles = np.linspace(0, 2 * np.pi, len(radar_cats), endpoint=False).tolist()
    angles += angles[:1]
    radar_models = [
        ("Logistic Regression", "#1D3557"),
        ("Random Forest", "#2A9D8F"),
        ("XGBoost", "#F4A261"),
        ("Weighted Soft-Vote Ensemble", "#E63946"),
        ("Stacking Ensemble", "#6A4C93"),
    ]
    for rname, rcol in radar_models:
        if rname in test_metrics_f1:
            vals = [test_metrics_f1[rname][k] for k in radar_keys]
            vals += vals[:1]
            lw = 2.6 if "Ensemble" in rname or rname == champion_name else 1.7
            ax_radar.plot(angles, vals, label=rname, color=rcol, linewidth=lw)
            ax_radar.fill(angles, vals, color=rcol, alpha=0.08)
    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(radar_cats, fontsize=9, fontweight="bold")
    ax_radar.set_ylim(0, 1.0)
    ax_radar.set_title("Multi-Metric Radar Comparison (Holdout)", fontweight="bold", pad=18)
    ax_radar.legend(loc="upper right", bbox_to_anchor=(1.32, 1.15), fontsize=8)

    # 8B: Ensemble Base-Model Diversity Correlation Heatmap
    ax_div = fig.add_subplot(1, 3, 2)
    sns.heatmap(corr_oof, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0.55, vmax=1.0, ax=ax_div)
    ax_div.set_title("Ensemble Diversity: Base Model OOF Correlation", fontweight="bold")

    # 8C: Learned Soft-Vote Weights vs Stacking Meta-Coefficients
    ax_wt = fig.add_subplot(1, 3, 3)
    ens_keys = list(soft_vote_weights.keys())
    ex = np.arange(len(ens_keys))
    sw_vals = [soft_vote_weights[k] for k in ens_keys]
    st_raw = np.array([max(stacking_meta_coefs.get(k, 0.0), 0.0) for k in ens_keys])
    st_norm = (st_raw / (st_raw.sum() + 1e-12)).tolist()
    ax_wt.bar(ex - 0.18, [v * 100 for v in sw_vals], 0.36, label="Soft-Vote Weight (%)", color="#1D3557")
    ax_wt.bar(ex + 0.18, [v * 100 for v in st_norm], 0.36, label="Stacking Positive Share (%)", color="#E76F51")
    ax_wt.set_xticks(ex)
    ax_wt.set_xticklabels([k.upper() for k in ens_keys], fontweight="bold")
    ax_wt.set_ylabel("Relative Contribution (%)")
    ax_wt.set_title("Learned Ensemble Weights (Soft-Vote vs Stacking)", fontweight="bold")
    ax_wt.legend(fontsize=8.5)

    fig.suptitle("08 | Model Radar Comparison, Ensemble Diversity & Learned Meta-Weights", fontsize=15, fontweight="bold", y=1.02)
    fig.tight_layout()
    _save_fig(fig, "08_model_comparison_radar_and_ensemble.png", "Radar chart, OOF diversity matrix & ensemble weights")

    # -------------------------------------------------------------------------
    # PLOT 09: ROC Curves, Precision-Recall Curves & Threshold Optimization Curve
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(18.5, 5.8))
    palette = sns.color_palette("tab10", len(test_probs))
    for (name, probs), col in zip(test_probs.items(), palette):
        lw = 2.7 if name == champion_name or "Ensemble" in name else 1.5
        ls = "-" if name == champion_name or "Ensemble" in name else "--"
        fpr, tpr, _ = roc_curve(y_test, probs)
        axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_test, probs):.4f})", color=col, linewidth=lw, linestyle=ls)

        prec, rec, _ = precision_recall_curve(y_test, probs)
        axes[1].plot(rec, prec, label=f"{name} ({average_precision_score(y_test, probs):.4f})", color=col, linewidth=lw, linestyle=ls)

    axes[0].plot([0, 1], [0, 1], "k:", alpha=0.6)
    axes[0].set_title("Holdout Test ROC Curves (Probabilities)", fontweight="bold")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].legend(loc="lower right", fontsize=7.8)

    prev = float(y_test.mean())
    axes[1].axhline(prev, color="k", linestyle=":", alpha=0.6, label=f"No-Skill ({prev:.3f})")
    axes[1].set_title("Holdout Test Precision-Recall Curves", fontweight="bold")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].legend(loc="upper right", fontsize=7.8)

    # 9C: Leak-Free OOF Decision Threshold Trade-Off Curve for Champion
    thrs = np.linspace(0.05, 0.85, 161)
    champ_oof = oof_probs[champion_name]
    p_list, r_list, f1_list, f15_list = [], [], [], []
    for t in thrs:
        pred_t = (champ_oof >= t).astype(int)
        p_list.append(precision_score(y_train, pred_t, zero_division=0))
        r_list.append(recall_score(y_train, pred_t, zero_division=0))
        f1_list.append(f1_score(y_train, pred_t, zero_division=0))
        f15_list.append(fbeta_score(y_train, pred_t, beta=1.5, zero_division=0))
    axes[2].plot(thrs, p_list, label="Precision", color="#2E86AB", linewidth=1.8)
    axes[2].plot(thrs, r_list, label="Recall", color="#E63946", linewidth=1.8)
    axes[2].plot(thrs, f1_list, label="F1-Score", color="#2A9D8F", linewidth=2.3)
    axes[2].plot(thrs, f15_list, label="F1.5 (HR Recall)", color="#6A4C93", linewidth=2.0, linestyle="--")
    axes[2].axvline(opt_thr_f1[champion_name], color="#2A9D8F", linestyle=":", linewidth=1.8, label=f"Balanced Thr={opt_thr_f1[champion_name]:.3f}")
    axes[2].axvline(opt_thr_hr[champion_name], color="#6A4C93", linestyle=":", linewidth=1.8, label=f"HR Thr={opt_thr_hr[champion_name]:.3f}")
    axes[2].set_title(f"OOF Threshold Trade-Off ({champion_name})", fontweight="bold")
    axes[2].set_xlabel("Decision Probability Threshold")
    axes[2].set_ylabel("Score")
    axes[2].legend(fontsize=7.8, loc="lower left")

    fig.suptitle("09 | ROC Curves, Precision-Recall Curves & Decision Threshold Optimization", fontsize=15, fontweight="bold", y=1.01)
    fig.tight_layout()
    _save_fig(fig, "09_roc_pr_and_threshold_curves.png", "ROC curves, PR curves & OOF threshold sweep")

    # -------------------------------------------------------------------------
    # PLOT 10: Side-by-Side Confusion Matrix Heatmaps Across All Key Models (2x3)
    # -------------------------------------------------------------------------
    cm_models = [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "Core-3 Voting Ensemble",
        "Weighted Soft-Vote Ensemble",
        "Stacking Ensemble",
    ]
    fig, axes = plt.subplots(2, 3, figsize=(17, 10.5))
    for ax, mname in zip(axes.flat, cm_models):
        md = test_metrics_f1[mname]
        cm_mat = np.array([[md["tn"], md["fp"]], [md["fn"], md["tp"]]])
        tot = cm_mat.sum()
        annots = np.array([[f"{cm_mat[i, j]}\n({cm_mat[i, j] / tot * 100:.1f}%)" for j in range(2)] for i in range(2)])
        sns.heatmap(
            cm_mat, annot=annots, fmt="", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Pred Stay (0)", "Pred Leave (1)"],
            yticklabels=["Actual Stay (0)", "Actual Leave (1)"],
            annot_kws={"size": 11, "weight": "bold"},
        )
        star_tag = " ★" if mname == champion_name else ""
        ax.set_title(
            f"{mname}{star_tag}\nThr={md['threshold']:.3f} | Rec={md['recall']:.1%} | Prec={md['precision']:.1%} | F1={md['f1']:.3f}",
            fontweight="bold", fontsize=9.8,
        )
    fig.suptitle("10 | Confusion Matrix Comparison Across Core Models & Ensembles (Holdout Test)", fontsize=15, fontweight="bold", y=0.995)
    fig.tight_layout()
    _save_fig(fig, "10_confusion_matrices_comparison.png", "2x3 confusion matrices for all core models & ensembles")

    # -------------------------------------------------------------------------
    # PLOT 11: Probability Calibration Reliability Curve & Subgroup Fairness Audit
    # -------------------------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16.5, 6.2))
    for cname in ["Logistic Regression", "Random Forest", "XGBoost", "Weighted Soft-Vote Ensemble", "Stacking Ensemble"]:
        if cname in test_probs:
            p_true, p_pred = calibration_curve(y_test, test_probs[cname], n_bins=8, strategy="quantile")
            ax1.plot(p_pred, p_true, marker="o", linewidth=2.0, label=f"{cname} (Brier={test_metrics_f1[cname]['brier']:.3f})")
    ax1.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Perfect Calibration")
    ax1.set_title("Probability Calibration Reliability Curve (Holdout Test)", fontweight="bold")
    ax1.set_xlabel("Mean Predicted Attrition Probability")
    ax1.set_ylabel("Observed Attrition Fraction")
    ax1.legend(fontsize=8.5)

    sg_labels, sg_aucs, sg_recs = [], [], []
    for col_name, entries in subgroup_report.items():
        for entry in entries:
            if entry["roc_auc"] is not None:
                sg_labels.append(f"{col_name}={entry['group']} (n={entry['n']})")
                sg_aucs.append(entry["roc_auc"])
                sg_recs.append(entry["recall"])
    y_pos = np.arange(len(sg_labels))
    ax2.barh(y_pos - 0.18, sg_aucs, 0.36, label="Subgroup ROC-AUC", color="#1D3557")
    ax2.barh(y_pos + 0.18, sg_recs, 0.36, label="Subgroup Recall (HR Mode)", color="#2A9D8F")
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(sg_labels, fontsize=8.8)
    ax2.set_xlim(0, 1.08)
    ax2.set_xlabel("Score")
    ax2.set_title(f"Subgroup Fairness & Performance Audit ({champion_name})", fontweight="bold")
    ax2.legend(fontsize=8.5)

    fig.suptitle("11 | Probability Calibration & Holdout Subgroup Fairness Audit", fontsize=15, fontweight="bold", y=1.01)
    fig.tight_layout()
    _save_fig(fig, "11_calibration_and_subgroup_audit.png", "Reliability curves & Department/OverTime/Gender subgroup audit")

    ui.subheader(f"Generated {len(saved_catalog)} Feature EDA & Model Comparison Graphs in `{plots_dir}`")
    ui.table(
        ["Plot File", "Description"],
        [[Path(p).name, d] for p, d in saved_catalog],
        ["<", "<"],
    )

    if show_plots:
        ui.info("Displayed all 11 plots inline and saved PNGs to disk.")

    return saved_catalog

## Execute Stages 1 & 2 — From-Scratch EDA, Split, Hardcoded Class Weights (`w0`, `w1`) & Fold Caches

In [ ]:
ui = TerminalUI()
np.random.seed(args.seed)

data_path = Path(args.data)
models_dir = Path(args.models_dir)
reports_dir = Path(args.reports_dir)
plots_dir = reports_dir / "plots"
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

ui.banner(
    "TALENTPULSE ML PIPELINE  |  FROM-SCRATCH EDA + HPO + ENSEMBLE LEARNING + PLOTS",
    f"Run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Python {platform.python_version()} | "
    f"scikit-learn {sklearn.__version__} | XGBoost {xgb_lib.__version__} | CPU Cores: {os.cpu_count()}",
)

if not data_path.exists():
    raise FileNotFoundError(f"Raw dataset not found at {data_path}")

df_raw = pd.read_csv(data_path)

# -------------------------------------------------------------------------
# STAGE 1: FROM-SCRATCH EDA
# -------------------------------------------------------------------------
eda_summary = run_eda_stage(
    df=df_raw,
    raw_path=data_path,
    legacy_clean_path=data_path.parent / "cleaned_employee_attrition.csv",
    ui=ui,
)

# -------------------------------------------------------------------------
# STAGE 2: STRICT LEAK-FREE SPLIT & FEATURE ENGINEERING SETUP
# -------------------------------------------------------------------------
ui.stage(2, 7, "Leak-Free Stratified Split & Domain Feature Engineering Pipeline")

X_raw = df_raw.drop(columns=["Attrition"])
y_all = (df_raw["Attrition"] == "Yes").astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y_all,
    test_size=0.20,
    random_state=args.seed,
    stratify=y_all,
)
ui.info(f"Train Split      : {X_train.shape[0]:4d} rows (Attrition=Yes: {int(y_train.sum()):3d} | {y_train.mean()*100:.2f}%)")
ui.info(f"Holdout Test     : {X_test.shape[0]:4d} rows (Attrition=Yes: {int(y_test.sum()):3d} | {y_test.mean()*100:.2f}%)  [UNTOUCHED until Stage 6]")

# Calculating balanced weights of each class strictly from training split
w0 = float(X_train.shape[0] / (2 * int((y_train == 0).sum())))  # weight of class No (0)
w1 = float(X_train.shape[0] / (2 * int((y_train == 1).sum())))  # weight of class Yes (1)
hardcoded_class_weight = {0: w0, 1: w1}
ui.info(
    f"Class Weights    : w0 (No=0) = {w0:.4f} | w1 (Yes=1) = {w1:.4f}  "
    f"[Hardcoded for Logistic Regression & Random Forest]"
)

preprocessor_template = build_preprocessor(X_train, drop_rate_columns=True)
pre_preview = clone(preprocessor_template).fit(X_train, y_train)
feature_names = list(pre_preview.named_steps["column_transformer"].get_feature_names_out())
fe_step = pre_preview.named_steps["feature_engineer"]
ui.info(
    f"Engineered Schema: {X_raw.shape[1]} raw cols -> {len(fe_step.output_columns_)} domain cols "
    f"({len(fe_step.numeric_columns_)} numeric + {len(fe_step.categorical_columns_)} categorical) -> "
    f"{len(feature_names)} model features after Yeo-Johnson PowerTransformer + OneHotEncoder"
)

hpo_fold_cache = build_leak_free_fold_cache(
    X_train, y_train, preprocessor_template, n_splits=args.hpo_folds, seed=args.seed, ui=ui, label="HPO"
)
oof_fold_cache = build_leak_free_fold_cache(
    X_train, y_train, preprocessor_template, n_splits=args.oof_folds, seed=args.seed, ui=ui, label="OOF Evaluation"
)

## Execute Stage 3 — Hyperparameter Optimization (6 Diverse Model Families)

In [ ]:
# -------------------------------------------------------------------------
# STAGE 3: HYPERPARAMETER OPTIMIZATION (6 DIVERSE MODEL FAMILIES)
# -------------------------------------------------------------------------
ui.stage(3, 7, f"Hyperparameter Optimization (HPO) Across 6 Diverse Model Families ({args.n_iter} Trials Each)")

tuned_estimators: Dict[str, BaseEstimator] = {}
hpo_best_params: Dict[str, Dict[str, Any]] = {}
hpo_best_scores: Dict[str, Dict[str, float]] = {}

# 3.1 Logistic Regression (CV-Tuned L2 / ElasticNet / L1 with Hardcoded Class Weights)
def lr_validator(p: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(p)
    out["solver"] = "lbfgs" if out["l1_ratio"] == 0.0 else "saga"
    return out

lr_space = {
    "C": loguniform(0.008, 1.5),
    "l1_ratio": [0.0, 0.0, 0.0, 0.15, 0.35, 0.65, 1.0],
}
tuned_estimators["Logistic Regression"], hpo_best_params["Logistic Regression"], hpo_best_scores["Logistic Regression"] = run_live_hpo(
    model_name="Logistic Regression",
    base_estimator=LogisticRegression(
        class_weight=hardcoded_class_weight,
        max_iter=3000,
        random_state=args.seed,
    ),
    param_distributions=lr_space,
    fold_cache=hpo_fold_cache,
    n_iter=args.n_iter,
    seed=args.seed + 1,
    ui=ui,
    param_validator=lr_validator,
)
hpo_best_params["Logistic Regression"]["class_weight"] = {"0": round(w0, 4), "1": round(w1, 4)}

# 3.2 Random Forest (Bagged Decision Trees with Hardcoded Class Weights; n_jobs=1 inside parallelized folds)
rf_space = {
    "n_estimators": randint(200, 520),
    "max_depth": [5, 6, 8, 10, 12, 16, None],
    "min_samples_split": randint(2, 16),
    "min_samples_leaf": randint(1, 9),
    "max_features": ["sqrt", "log2", 0.25, 0.35, 0.5],
    "criterion": ["gini", "entropy", "log_loss"],
}
tuned_estimators["Random Forest"], hpo_best_params["Random Forest"], hpo_best_scores["Random Forest"] = run_live_hpo(
    model_name="Random Forest",
    base_estimator=RandomForestClassifier(
        class_weight=hardcoded_class_weight,
        random_state=args.seed,
        n_jobs=1,
    ),
    param_distributions=rf_space,
    fold_cache=hpo_fold_cache,
    n_iter=args.n_iter,
    seed=args.seed + 2,
    ui=ui,
)
hpo_best_params["Random Forest"]["class_weight"] = {"0": round(w0, 4), "1": round(w1, 4)}

# 3.3 XGBoost (Regularized Gradient Boosted Trees)
xgb_space = {
    "n_estimators": randint(180, 580),
    "learning_rate": loguniform(0.01, 0.085),
    "max_depth": [2, 3, 4, 5],
    "min_child_weight": randint(2, 12),
    "subsample": uniform(0.60, 0.35),
    "colsample_bytree": uniform(0.35, 0.50),
    "gamma": uniform(0.0, 2.5),
    "reg_alpha": loguniform(0.05, 8.0),
    "reg_lambda": loguniform(0.5, 15.0),
    "scale_pos_weight": [1.0, 1.0, 1.5, 2.0, 3.0],
}
tuned_estimators["XGBoost"], hpo_best_params["XGBoost"], hpo_best_scores["XGBoost"] = run_live_hpo(
    model_name="XGBoost",
    base_estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=args.seed,
        n_jobs=1,
    ),
    param_distributions=xgb_space,
    fold_cache=hpo_fold_cache,
    n_iter=args.n_iter,
    seed=args.seed + 3,
    ui=ui,
)

# 3.4 Extra Trees (Low-Variance Bagging Diversity)
et_space = {
    "n_estimators": randint(200, 520),
    "max_depth": [6, 8, 10, 14, 18, None],
    "min_samples_split": randint(2, 15),
    "min_samples_leaf": randint(1, 8),
    "max_features": ["sqrt", "log2", 0.35, 0.55],
    "criterion": ["gini", "entropy", "log_loss"],
    "class_weight": [None, None, {0: 1.0, 1: 2.0}, "balanced_subsample"],
}
tuned_estimators["Extra Trees"], hpo_best_params["Extra Trees"], hpo_best_scores["Extra Trees"] = run_live_hpo(
    model_name="Extra Trees",
    base_estimator=ExtraTreesClassifier(random_state=args.seed, n_jobs=1),
    param_distributions=et_space,
    fold_cache=hpo_fold_cache,
    n_iter=max(20, args.n_iter - 5),
    seed=args.seed + 4,
    ui=ui,
)

# 3.5 Stochastic Gradient Boosting (Sklearn GBDT Diversity)
gb_space = {
    "n_estimators": randint(120, 420),
    "learning_rate": loguniform(0.012, 0.08),
    "max_depth": [2, 3, 4],
    "min_samples_leaf": randint(3, 18),
    "subsample": uniform(0.60, 0.35),
    "max_features": ["sqrt", "log2", 0.35, 0.6],
}
tuned_estimators["Gradient Boosting"], hpo_best_params["Gradient Boosting"], hpo_best_scores["Gradient Boosting"] = run_live_hpo(
    model_name="Gradient Boosting",
    base_estimator=GradientBoostingClassifier(random_state=args.seed),
    param_distributions=gb_space,
    fold_cache=hpo_fold_cache,
    n_iter=max(20, args.n_iter - 5),
    seed=args.seed + 5,
    ui=ui,
)

# 3.6 Calibrated RBF-SVC (Non-Linear Kernel Margin Diversity, sklearn 1.9 compliant)
svc_space = {
    "estimator__C": loguniform(0.15, 8.0),
    "estimator__gamma": ["scale", "auto", 0.005, 0.01, 0.02, 0.04],
    "estimator__class_weight": [None, None, {0: 1.0, 1: 2.0}, "balanced"],
}
tuned_estimators["Calibrated RBF-SVC"], hpo_best_params["Calibrated RBF-SVC"], hpo_best_scores["Calibrated RBF-SVC"] = run_live_hpo(
    model_name="Calibrated RBF-SVC",
    base_estimator=CalibratedClassifierCV(
        estimator=SVC(kernel="rbf", random_state=args.seed),
        cv=5,
        method="sigmoid",
        ensemble=False,
    ),
    param_distributions=svc_space,
    fold_cache=hpo_fold_cache,
    n_iter=max(18, args.n_iter - 8),
    seed=args.seed + 6,
    ui=ui,
)

## Execute Stages 4 & 5 — Ensemble Learning (Soft-Vote & Stacking) + OOF Threshold Tuning

In [ ]:
# -------------------------------------------------------------------------
# STAGE 4: ENSEMBLE LEARNING (WEIGHTED SOFT-VOTING & STACKING) + DIVERSITY
# -------------------------------------------------------------------------
ui.stage(4, 7, f"Ensemble Learning & {args.oof_folds}-Fold Out-Of-Fold Cross-Validation")

base_keys = list(tuned_estimators.keys())
oof_probs: Dict[str, np.ndarray] = {}
oof_stats: Dict[str, Dict[str, float]] = {}

ui.subheader(f"Evaluating All 6 HPO-Tuned Base Models ({args.oof_folds}-Fold Leak-Free OOF CV)")
for name in base_keys:
    oof_probs[name], oof_stats[name] = evaluate_oof_from_cache(
        model_name=name,
        estimator=tuned_estimators[name],
        fold_cache=oof_fold_cache,
        n_samples=len(y_train),
        ui=ui,
    )

# Diversity Correlation Matrix across base model OOF predictions
ui.subheader("Ensemble Diversity Audit: Pairwise OOF Probability Correlation Matrix")
short_names = {
    "Logistic Regression": "LR",
    "Random Forest": "RF",
    "XGBoost": "XGB",
    "Extra Trees": "ET",
    "Gradient Boosting": "GB",
    "Calibrated RBF-SVC": "SVC",
}
oof_df = pd.DataFrame({short_names[k]: oof_probs[k] for k in base_keys})
corr_oof = oof_df.corr()
corr_headers = ["Model"] + list(corr_oof.columns)
corr_rows = [[idx] + [f"{corr_oof.loc[idx, c]:.3f}" for c in corr_oof.columns] for idx in corr_oof.index]
ui.table(corr_headers, corr_rows, ["<"] + [">"] * len(corr_oof.columns))

# Build Ensembles
core3_estimators = [
    ("lr", clone(tuned_estimators["Logistic Regression"])),
    ("rf", clone(tuned_estimators["Random Forest"])),
    ("xgb", clone(tuned_estimators["XGBoost"])),
]
core3_voting = WeightedSoftVotingClassifier(
    estimators=core3_estimators, cv=5, l2_diversity_penalty=0.015, random_state=args.seed
)

all6_estimators = [(short_names[k].lower(), clone(tuned_estimators[k])) for k in base_keys]
full_voting = WeightedSoftVotingClassifier(
    estimators=all6_estimators, cv=5, l2_diversity_penalty=0.015, random_state=args.seed
)

# Tune Stacking Classifier Meta-Learner Regularization C via Leak-Free CV
ui.subheader("Tuning Stacking Ensemble Meta-Learner (Regularization C Sweep via Leak-Free CV)")
best_meta_c, best_stack_obj = 0.5, -1e9
for meta_c in [0.05, 0.15, 0.35, 0.75, 1.5]:
    cand_stack = StackingClassifier(
        estimators=all6_estimators,
        final_estimator=LogisticRegression(C=meta_c, l1_ratio=0.0, max_iter=2000, random_state=args.seed),
        cv=5,
        stack_method="predict_proba",
        n_jobs=1,
    )
    fold_res = Parallel(n_jobs=min(len(hpo_fold_cache), os.cpu_count() or 1), prefer="threads")(
        delayed(_fit_and_score_fold)(cand_stack, {}, X_f_tr, y_f_tr, X_f_val, y_f_val)
        for X_f_tr, y_f_tr, X_f_val, y_f_val, _ in hpo_fold_cache
    )
    m_auc = float(np.mean([r[0] for r in fold_res]))
    m_pr = float(np.mean([r[1] for r in fold_res]))
    m_ll = float(np.mean([r[2] for r in fold_res]))
    obj = 0.55 * m_auc + 0.35 * m_pr - 0.10 * m_ll
    mark = f"{ui.GREEN}{ui.STAR}{ui.RESET}" if obj > best_stack_obj else " "
    ui.info(f"  {mark} Meta-LR C={meta_c:4.2f} | 5-Fold ROC-AUC={m_auc:.4f} | PR-AUC={m_pr:.4f} | LogLoss={m_ll:.4f}")
    if obj > best_stack_obj:
        best_stack_obj = obj
        best_meta_c = meta_c

stacking_clf = StackingClassifier(
    estimators=all6_estimators,
    final_estimator=LogisticRegression(C=best_meta_c, l1_ratio=0.0, max_iter=2000, random_state=args.seed),
    cv=5,
    stack_method="predict_proba",
    n_jobs=1,
)
hpo_best_params["Stacking Ensemble"] = {"meta_estimator": "LogisticRegression", "meta_C": best_meta_c, "cv": 5}

ensemble_models: Dict[str, BaseEstimator] = {
    "Core-3 Voting Ensemble": core3_voting,
    "Weighted Soft-Vote Ensemble": full_voting,
    "Stacking Ensemble": stacking_clf,
}

ui.subheader(f"Evaluating Ensemble Architectures ({args.oof_folds}-Fold Leak-Free OOF CV)")
for ename, eclf in ensemble_models.items():
    oof_probs[ename], oof_stats[ename] = evaluate_oof_from_cache(
        model_name=ename,
        estimator=eclf,
        fold_cache=oof_fold_cache,
        n_samples=len(y_train),
        ui=ui,
        parallel_folds=False,
    )

# -------------------------------------------------------------------------
# STAGE 5: OUT-OF-FOLD DECISION THRESHOLD OPTIMIZATION (ZERO TEST LEAKAGE)
# -------------------------------------------------------------------------
ui.stage(5, 7, "Out-Of-Fold Decision Threshold Optimization (Balanced F1 & HR High-Recall)")

all_model_names = base_keys + list(ensemble_models.keys())
opt_thr_f1: Dict[str, float] = {}
opt_thr_hr: Dict[str, float] = {}
thr_rows = []
for name in all_model_names:
    t_f1, _ = optimize_threshold_oof(y_train, oof_probs[name], beta=1.0)
    t_hr, _ = optimize_threshold_oof(y_train, oof_probs[name], beta=1.5)
    opt_thr_f1[name] = t_f1
    opt_thr_hr[name] = t_hr
    m_f1 = compute_metrics(y_train, oof_probs[name], threshold=t_f1)
    m_hr = compute_metrics(y_train, oof_probs[name], threshold=t_hr)
    thr_rows.append(
        [
            name,
            f"{oof_stats[name]['oof_roc_auc']:.4f}",
            f"{oof_stats[name]['oof_pr_auc']:.4f}",
            f"{t_f1:.3f}",
            f"{m_f1['f1']:.4f} (P={m_f1['precision']:.2f}, R={m_f1['recall']:.2f})",
            f"{t_hr:.3f}",
            f"{m_hr['recall']:.4f} (P={m_hr['precision']:.2f}, F1={m_hr['f1']:.3f})",
        ]
    )

ui.table(
    ["Model", "OOF AUC", "OOF PR", "Thr(F1)", "OOF Balanced F1", "Thr(HR)", "OOF High-Recall Mode"],
    thr_rows,
    ["<", ">", ">", ">", "<", ">", "<"],
)

def oof_composite(mname: str) -> float:
    st = oof_stats[mname]
    return 0.55 * st["oof_roc_auc"] + 0.35 * st["oof_pr_auc"] - 0.10 * st["oof_log_loss"]

champion_name = max(all_model_names, key=oof_composite)
ui.success(
    f"CV-Selected Champion (strictly pre-test): {champion_name} "
    f"(OOF ROC-AUC={oof_stats[champion_name]['oof_roc_auc']:.4f}, "
    f"OOF PR-AUC={oof_stats[champion_name]['oof_pr_auc']:.4f}, "
    f"OOF LogLoss={oof_stats[champion_name]['oof_log_loss']:.4f})"
)

## Execute Stage 6 — Final Fit, Untouched Holdout Evaluation & Subgroup Fairness Audit

In [ ]:
# -------------------------------------------------------------------------
# STAGE 6: FINAL FIT ON FULL TRAIN & UNTOUCHED HOLDOUT TEST EVALUATION
# -------------------------------------------------------------------------
ui.stage(6, 7, "Final Training & Untouched Holdout Test Set Evaluation + Subgroup Audit")

final_preprocessor = clone(preprocessor_template)
X_train_proc = final_preprocessor.fit_transform(X_train, y_train)
X_test_proc = final_preprocessor.transform(X_test)
final_feature_names = list(
    final_preprocessor.named_steps["column_transformer"].get_feature_names_out()
)

fitted_models: Dict[str, BaseEstimator] = {}
train_probs: Dict[str, np.ndarray] = {}
test_probs: Dict[str, np.ndarray] = {}
train_metrics_f1: Dict[str, Dict[str, Any]] = {}
test_metrics_f1: Dict[str, Dict[str, Any]] = {}
test_metrics_hr: Dict[str, Dict[str, Any]] = {}

all_estimators_map: Dict[str, BaseEstimator] = {**tuned_estimators, **ensemble_models}
for name in all_model_names:
    t0 = time.perf_counter()
    est = clone(all_estimators_map[name])
    est.fit(X_train_proc, y_train)
    fitted_models[name] = est
    train_probs[name] = est.predict_proba(X_train_proc)[:, 1]
    test_probs[name] = est.predict_proba(X_test_proc)[:, 1]
    train_metrics_f1[name] = compute_metrics(y_train, train_probs[name], threshold=opt_thr_f1[name])
    test_metrics_f1[name] = compute_metrics(y_test, test_probs[name], threshold=opt_thr_f1[name])
    test_metrics_hr[name] = compute_metrics(y_test, test_probs[name], threshold=opt_thr_hr[name])
    ui.info(f"Fitted final {name:28s} in {time.perf_counter() - t0:5.2f}s")

# Show learned ensemble weights & stacking meta-coefficients
ui.subheader("Learned Ensemble Weights (Weighted Soft-Vote) & Stacking Meta-Coefficients")
wv_fitted: WeightedSoftVotingClassifier = fitted_models["Weighted Soft-Vote Ensemble"]  # type: ignore
for m_key, weight in sorted(wv_fitted.weight_map_.items(), key=lambda x: x[1], reverse=True):
    bar = ui.progress_bar(int(round(weight * 100)), 100, width=22)
    ui.info(f"Soft-Vote Weight [{m_key.upper():4s}] : {bar} {weight*100:5.1f}%")

st_fitted: StackingClassifier = fitted_models["Stacking Ensemble"]  # type: ignore
meta_coefs_arr = st_fitted.final_estimator_.coef_[0]
stacking_meta_coefs: Dict[str, float] = {
    name: float(coef) for (name, _), coef in zip(st_fitted.estimators, meta_coefs_arr)
}
meta_str = ", ".join(f"{name.upper()}={coef:+.3f}" for name, coef in stacking_meta_coefs.items())
ui.info(f"Stacking Meta-LR Coefs : {meta_str} (intercept={st_fitted.final_estimator_.intercept_[0]:+.3f})")

# Holdout Test Leaderboard — Balanced F1 Mode
ui.subheader("Untouched Holdout Test Leaderboard (Balanced F1 OOF Threshold)")
lb_rows = []
for name in all_model_names:
    tm = test_metrics_f1[name]
    trm = train_metrics_f1[name]
    gap = trm["roc_auc"] - tm["roc_auc"]
    tag = f"{ui.STAR} " if name == champion_name else "  "
    lb_rows.append(
        [
            f"{tag}{name}",
            f"{oof_stats[name]['oof_roc_auc']:.4f}",
            f"{tm['roc_auc']:.4f}",
            f"{tm['pr_auc']:.4f}",
            f"{tm['accuracy']:.4f}",
            f"{tm['precision']:.4f}",
            f"{tm['recall']:.4f}",
            f"{tm['f1']:.4f}",
            f"{gap:+.4f}",
            f"{tm['tp']}/{tm['tp']+tm['fn']} (FN={tm['fn']})",
        ]
    )
ui.table(
    ["Model", "CV AUC", "Test AUC", "PR-AUC", "Acc", "Prec", "Recall", "F1", "Gap", "Caught (FN)"],
    lb_rows,
    ["<", ">", ">", ">", ">", ">", ">", ">", ">", ">"],
)

# Holdout Test Leaderboard — HR High-Recall Mode
ui.subheader("Untouched Holdout Test Leaderboard (HR High-Recall Mode: F1.5 OOF Threshold)")
hr_rows = []
for name in all_model_names:
    hm = test_metrics_hr[name]
    tag = f"{ui.STAR} " if name == champion_name else "  "
    hr_rows.append(
        [
            f"{tag}{name}",
            f"{hm['threshold']:.3f}",
            f"{hm['roc_auc']:.4f}",
            f"{hm['accuracy']:.4f}",
            f"{hm['precision']:.4f}",
            f"{hm['recall']:.4f}",
            f"{hm['f1']:.4f}",
            f"{hm['tn']}/{hm['fp']}/{hm['fn']}/{hm['tp']}",
        ]
    )
ui.table(
    ["Model", "Thr", "Test AUC", "Acc", "Prec", "Recall", "F1", "TN/FP/FN/TP"],
    hr_rows,
    ["<", ">", ">", ">", ">", ">", ">", ">"],
)

# Direct Before vs After Improvement Table against Legacy Baseline
ui.subheader("Upgrade Impact: Legacy Saved Models (Baseline) vs New HPO + Ensemble Pipeline")
legacy_ref = {
    "Logistic Regression": {"roc_auc": 0.8088, "pr_auc": 0.6415, "f1": 0.4800, "recall": 0.7660, "acc": 0.7347},
    "Random Forest": {"roc_auc": 0.7895, "pr_auc": 0.4032, "f1": 0.4865, "recall": 0.5745, "acc": 0.8061},
    "XGBoost": {"roc_auc": 0.7882, "pr_auc": 0.4660, "f1": 0.4771, "recall": 0.5532, "acc": 0.8061},
}
cmp_rows = []
for mname, base in legacy_ref.items():
    new_m = test_metrics_f1[mname]
    cmp_rows.append(
        [
            mname,
            f"{base['roc_auc']:.4f} -> {new_m['roc_auc']:.4f} ({new_m['roc_auc']-base['roc_auc']:+.4f})",
            f"{base['pr_auc']:.4f} -> {new_m['pr_auc']:.4f} ({new_m['pr_auc']-base['pr_auc']:+.4f})",
            f"{base['f1']:.4f} -> {new_m['f1']:.4f} ({new_m['f1']-base['f1']:+.4f})",
            f"{base['acc']:.4f} -> {new_m['accuracy']:.4f} ({new_m['accuracy']-base['acc']:+.4f})",
        ]
    )
for ens_name in ["Weighted Soft-Vote Ensemble", "Stacking Ensemble"]:
    em = test_metrics_f1[ens_name]
    cmp_rows.append(
        [
            ens_name,
            f"0.8088 -> {em['roc_auc']:.4f} ({em['roc_auc']-0.8088:+.4f})",
            f"0.6415 -> {em['pr_auc']:.4f} ({em['pr_auc']-0.6415:+.4f})",
            f"0.4865 -> {em['f1']:.4f} ({em['f1']-0.4865:+.4f})",
            f"0.8061 -> {em['accuracy']:.4f} ({em['accuracy']-0.8061:+.4f})",
        ]
    )
ui.table(
    ["Model", "Test ROC-AUC (Delta)", "Test PR-AUC (Delta)", "Test F1 (Delta)", "Test Acc (Delta)"],
    cmp_rows,
    ["<", ">", ">", ">", ">"],
)

# Subgroup Fairness & Error Analysis on Holdout Test
ui.subheader(f"Holdout Subgroup & Fairness Audit ({champion_name} @ HR Threshold={opt_thr_hr[champion_name]:.3f})")
champ_p_test = test_probs[champion_name]
champ_pred_hr = (champ_p_test >= opt_thr_hr[champion_name]).astype(int)
subgroup_report: Dict[str, List[Dict[str, Any]]] = {}
sg_rows = []
for grp_col in ["Department", "OverTime", "Gender", "MaritalStatus"]:
    subgroup_report[grp_col] = []
    for gval, idxs in X_test.groupby(grp_col).groups.items():
        pos_mask = X_test.index.isin(idxs)
        y_sub = y_test[pos_mask]
        p_sub = champ_p_test[pos_mask]
        pred_sub = champ_pred_hr[pos_mask]
        auc_sub = float(roc_auc_score(y_sub, p_sub)) if len(np.unique(y_sub)) > 1 else float("nan")
        rec_sub = float(recall_score(y_sub, pred_sub, zero_division=0))
        prec_sub = float(precision_score(y_sub, pred_sub, zero_division=0))
        subgroup_report[grp_col].append(
            {
                "group": str(gval),
                "n": int(len(y_sub)),
                "actual_yes": int(y_sub.sum()),
                "roc_auc": None if np.isnan(auc_sub) else round(auc_sub, 4),
                "recall": round(rec_sub, 4),
                "precision": round(prec_sub, 4),
            }
        )
        sg_rows.append(
            [
                f"{grp_col}={gval}",
                f"{len(y_sub)}",
                f"{int(y_sub.sum())}",
                "N/A" if np.isnan(auc_sub) else f"{auc_sub:.4f}",
                f"{rec_sub:.3f}",
                f"{prec_sub:.3f}",
            ]
        )
ui.table(["Subgroup", "N", "Leavers", "ROC-AUC", "Recall", "Precision"], sg_rows, ["<", ">", ">", ">", ">", ">"])

## Execute Stage 7 — Save Models, Display All 11 Inline Plots & Smoke Verification

In [ ]:
# -------------------------------------------------------------------------
# STAGE 7: ARTIFACT EXPORT, 11-PLOT SUITE & AUTOMATED SMOKE VERIFICATION
# -------------------------------------------------------------------------
ui.stage(7, 7, "Saving Production Artifacts, Generating 11 Feature/Comparison Plots & Verification")

artifact_paths = {
    "scaler": models_dir / "scaler.pkl",
    "logistic_regression": models_dir / "logistic_regression_model.pkl",
    "random_forest": models_dir / "random_forest_model.pkl",
    "xgboost": models_dir / "xgb_model.pkl",
    "voting_ensemble": models_dir / "voting_ensemble_model.pkl",
    "stacking_ensemble": models_dir / "stacking_ensemble_model.pkl",
    "best_pipeline": models_dir / "best_pipeline.pkl",
}

joblib.dump(final_preprocessor, artifact_paths["scaler"])
joblib.dump(fitted_models["Logistic Regression"], artifact_paths["logistic_regression"])
joblib.dump(fitted_models["Random Forest"], artifact_paths["random_forest"])
joblib.dump(fitted_models["XGBoost"], artifact_paths["xgboost"])
joblib.dump(fitted_models["Weighted Soft-Vote Ensemble"], artifact_paths["voting_ensemble"])
joblib.dump(fitted_models["Stacking Ensemble"], artifact_paths["stacking_ensemble"])

best_full_pipeline = Pipeline(
    steps=[
        ("preprocessor", final_preprocessor),
        ("classifier", fitted_models[champion_name]),
    ]
)
joblib.dump(best_full_pipeline, artifact_paths["best_pipeline"])

for label, path in artifact_paths.items():
    size_kb = path.stat().st_size / 1024.0
    ui.success(f"Saved {label:22s} -> {path} ({size_kb:7.1f} KB)")

metadata = {
    "generated_at": datetime.now().isoformat(),
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb_lib.__version__,
        "joblib": joblib.__version__,
    },
    "config": {
        "seed": args.seed,
        "hpo_folds": args.hpo_folds,
        "oof_folds": args.oof_folds,
        "n_iter": args.n_iter,
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "engineered_feature_count": len(final_feature_names),
        "hardcoded_class_weight_lr_rf": {"0": round(w0, 6), "1": round(w1, 6)},
    },
    "champion_model": champion_name,
    "ensemble_weights": wv_fitted.weight_map_,
    "stacking_meta_coefs": stacking_meta_coefs,
    "optimal_thresholds": {
        name: {
            "balanced_f1": round(opt_thr_f1[name], 4),
            "hr_high_recall": round(opt_thr_hr[name], 4),
        }
        for name in all_model_names
    },
    "hpo_best_params": hpo_best_params,
    "oof_cv_metrics": oof_stats,
    "train_metrics_balanced_f1": train_metrics_f1,
    "test_metrics_balanced_f1": test_metrics_f1,
    "test_metrics_hr_recall": test_metrics_hr,
    "subgroup_audit": subgroup_report,
    "feature_names": final_feature_names,
}
meta_path = models_dir / "model_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
eda_path = reports_dir / "eda_summary.json"
eda_path.write_text(json.dumps(eda_summary, indent=2), encoding="utf-8")
ui.success(f"Saved model metadata       -> {meta_path}")
ui.success(f"Saved EDA report JSON      -> {eda_path}")

# Generate all 11 Feature EDA & Model Comparison Charts
if not args.skip_plots:
    df_engineered_all = final_preprocessor.named_steps["feature_engineer"].transform(X_raw)
    generate_all_plots(
        df_raw=df_raw,
        df_engineered_all=df_engineered_all,
        y_train=y_train,
        y_test=y_test,
        oof_probs=oof_probs,
        test_probs=test_probs,
        test_metrics_f1=test_metrics_f1,
        test_metrics_hr=test_metrics_hr,
        train_metrics_f1=train_metrics_f1,
        oof_stats=oof_stats,
        corr_oof=corr_oof,
        soft_vote_weights=wv_fitted.weight_map_,
        stacking_meta_coefs=stacking_meta_coefs,
        legacy_ref=legacy_ref,
        subgroup_report=subgroup_report,
        champion_name=champion_name,
        opt_thr_f1=opt_thr_f1,
        opt_thr_hr=opt_thr_hr,
        fitted_models=fitted_models,
        feature_names=final_feature_names,
        plots_dir=plots_dir,
        show_plots=args.show_plots,
        ui=ui,
    )

# Automated Artifact Reload & Inference Smoke Tests (Full Schema + Legacy app.py 18-Feature Schema)
ui.subheader("Automated Artifact Verification (Reload + Inference Smoke Checks)")
loaded_scaler = joblib.load(artifact_paths["scaler"])
loaded_lr = joblib.load(artifact_paths["logistic_regression"])
loaded_rf = joblib.load(artifact_paths["random_forest"])
loaded_xgb = joblib.load(artifact_paths["xgboost"])
loaded_vote = joblib.load(artifact_paths["voting_ensemble"])
loaded_stack = joblib.load(artifact_paths["stacking_ensemble"])
loaded_best = joblib.load(artifact_paths["best_pipeline"])

X_sample_proc = loaded_scaler.transform(X_test.iloc[:5])
for m_label, m_obj in [
    ("LogisticRegression", loaded_lr),
    ("RandomForest", loaded_rf),
    ("XGBoost", loaded_xgb),
    ("WeightedSoftVoting", loaded_vote),
    ("StackingEnsemble", loaded_stack),
]:
    p = m_obj.predict_proba(X_sample_proc)[:, 1]
    assert p.shape == (5,) and np.all(np.isfinite(p)) and np.all((p >= 0.0) & (p <= 1.0)), f"Bad output for {m_label}"

p_pipe = loaded_best.predict_proba(X_test.iloc[:5])[:, 1]
assert np.allclose(p_pipe, test_probs[champion_name][:5], atol=1e-6), "best_pipeline parity mismatch!"
ui.success("Full raw DataFrame inference parity verified across all 7 serialized artifacts.")

legacy_probe = np.array(
    [[35, 10, 3, 3, 2, 3, 5000, 2, 12, 3, 1, 10, 5, 3, 2, 3, 0, 1]], dtype=float
)
assert legacy_probe.shape == (1, len(LEGACY_18_FEATURES))
legacy_proc = loaded_scaler.transform(legacy_probe)
p_lr_leg = float(loaded_lr.predict_proba(legacy_proc)[0, 1])
p_rf_leg = float(loaded_rf.predict_proba(legacy_proc)[0, 1])
p_xgb_leg = float(loaded_xgb.predict_proba(legacy_proc)[0, 1])
p_ens_leg = float(loaded_vote.predict_proba(legacy_proc)[0, 1])
assert all(0.0 <= v <= 1.0 and np.isfinite(v) for v in [p_lr_leg, p_rf_leg, p_xgb_leg, p_ens_leg])
ui.success(
    f"Legacy 18-feature app.py compatibility verified -> "
    f"LR={p_lr_leg:.3f}, RF={p_rf_leg:.3f}, XGB={p_xgb_leg:.3f}, SoftVote={p_ens_leg:.3f}"
)

ui.banner(
    f"PIPELINE COMPLETED SUCCESSFULLY IN {ui.elapsed().strip()}  |  CHAMPION: {champion_name.upper()}",
    f"Holdout Test ROC-AUC: {test_metrics_f1[champion_name]['roc_auc']:.4f} | "
    f"PR-AUC: {test_metrics_f1[champion_name]['pr_auc']:.4f} | "
    f"F1 (Balanced): {test_metrics_f1[champion_name]['f1']:.4f} | "
    f"Recall (HR Mode): {test_metrics_hr[champion_name]['recall']:.4f}",
)